*Projective Orchestration Transformer*

lifts timbre-neutral pitch data to higher-dimensional orchestral timbre space

input and output as MusicXML


---
Cell 1 — Setup/Restore


In [ ]:
import os, sys, shutil, zipfile, warnings
warnings.filterwarnings('ignore')
from google.colab import files as colab_files

WORK_DIR = '/content/orchestration'
os.makedirs(WORK_DIR, exist_ok=True)
os.system('pip install -q music21 pretty_midi mido pyarrow')

def find_zip(keyword):
    for f in os.listdir('/content'):
        if keyword.lower() in f.lower() and f.endswith('.zip'):
            return f'/content/{f}'
    return None

# ── Upload / locate LOP dataset zip ──────────────────────────────────────
lop_zip = find_zip('AI-for-Projective')
if lop_zip is None:
    print('📂 Upload AI-for-Projective-Musical-Orchestration-main.zip')
    uploaded = colab_files.upload()
    lop_zip = f'/content/{list(uploaded.keys())[0]}'
print(f'  LOP zip: {lop_zip}')
with zipfile.ZipFile(lop_zip, 'r') as z:
    z.extractall(WORK_DIR)

# ── Upload / locate orchestranet code zip ────────────────────────────────
orch_zip = find_zip('orchestranet')
if orch_zip is None:
    print('📂 Upload orchestranet.zip')
    uploaded = colab_files.upload()
    orch_zip = f'/content/{list(uploaded.keys())[0]}'
print(f'  Code zip: {orch_zip}')
os.system(f'unzip -o "{orch_zip}" -d {WORK_DIR}')

# ── Locate roots ─────────────────────────────────────────────────────────
DATA_ROOT = CODE_ROOT = None
for item in os.listdir(WORK_DIR):
    c = os.path.join(WORK_DIR, item)
    if not os.path.isdir(c): continue
    if os.path.exists(os.path.join(c, 'data')): DATA_ROOT = c
    if os.path.exists(os.path.join(c, 'orchestrate.py')): CODE_ROOT = c

RUNS_DIR = os.path.join(CODE_ROOT, 'runs/orchestranet_v6')
os.makedirs(RUNS_DIR, exist_ok=True)
sys.path.insert(0, CODE_ROOT)
sys.path.insert(0, os.path.join(CODE_ROOT, 'src'))
os.chdir(CODE_ROOT)
print(f'  DATA_ROOT: {DATA_ROOT}')
print(f'  CODE_ROOT: {CODE_ROOT}')

# ── Upload checkpoint (optional — skip if training from scratch) ──────────
best_ckpt = os.path.join(RUNS_DIR, 'best.pt')
if not os.path.exists(best_ckpt):
    print('\n📂 Upload v6_best.pt  (skip with Cancel if training from scratch)')
    try:
        uploaded = colab_files.upload()
        if uploaded:
            fname = list(uploaded.keys())[0]
            # find where it landed
            import glob
            candidates = glob.glob(f'/content/**/{fname}', recursive=True)
            src = candidates[0] if candidates else f'/content/{fname}'
            shutil.copy2(src, best_ckpt)
            print(f'  ✅ Checkpoint saved to {best_ckpt}')
        else:
            print('  ⚠️  No checkpoint uploaded — inference will not work until trained')
    except Exception as e:
        print(f'  ⚠️  Checkpoint upload skipped: {e}')
else:
    print(f'  ✅ Checkpoint already present: {best_ckpt}')

# ── Build orch_pitch targets from raw MIDI if missing ────────────────────
import numpy as np, pandas as pd, pretty_midi, glob as _glob
from pathlib import Path
from src.model import INST_NAMES, N_ORCH, N_PITCH

HOP_S = 0.05

NAME_TO_ORCH_NAME = {
    'flute 1':'flute','flute 2':'flute','flute 3':'flute',
    'flute picc 1':'flute','piccolo':'flute','piccolo 1':'flute',
    'oboe 1':'oboe','oboe 2':'oboe','oboe 3':'oboe','english horn':'oboe',
    'clarinet 1 (bb)':'clarinet','clarinet 2 (bb)':'clarinet','clarinet 3 (bb)':'clarinet',
    'clarinet in bb 1':'clarinet','clarinet bass 1':'clarinet','bass clarinet':'clarinet',
    'bassoon 1':'bassoon','bassoon 2':'bassoon','bassoon 3':'bassoon',
    'contrabassoon':'bassoon','bassoon contra 1':'bassoon',
    'horn 1':'horn','horn 2':'horn','horn 3':'horn','horn 4':'horn',
    'trumpet 1':'trumpet','trumpet 2':'trumpet','trumpet 3':'trumpet',
    'trombone 1':'trombone','trombone 2':'trombone','trombone 3':'trombone',
    'trombone bass':'trombone','bass trb (tuba mb)':'trombone',
    'tuba':'tuba','tuba 1':'tuba','tuba 1 (harp mb)':'tuba',
    'violins i tutti':'violin_1','violin i tutti':'violin_1',
    'violin i solo':'violin_1','violin i solo 1':'violin_1',
    'violin i solo 2':'violin_1','violins ib':'violin_1',
    'violins ii tutti':'violin_2','violin ii tutti':'violin_2',
    'violin ii solo':'violin_2','violin ii solo 1':'violin_2',
    'violin ii solo 2':'violin_2','violins iib':'violin_2',
    'viola tutti':'viola','viola b':'viola',
    'viola solo 1':'viola','viola solo 2':'viola',
    'celli tutti':'cello','cello tutti':'cello',
    'cello solo 1':'cello','cello solo 2':'cello',
    'double bass tutti':'double_bass','double bass solo':'double_bass',
    'double bass solo 1':'double_bass',
    'timpani 1':'timpani','timpani (hn5 full)':'timpani',
    'harp 1':'harp','harp 2':'harp',
}
NAME_TO_IDX = {k: INST_NAMES.index(v) for k,v in NAME_TO_ORCH_NAME.items() if v in INST_NAMES}

def match(n):
    n = n.lower().strip()
    if n in NAME_TO_IDX: return NAME_TO_IDX[n]
    for k, i in NAME_TO_IDX.items():
        if k in n or n in k: return i
    return -1

def build_targets(midi_path, T):
    pm = pretty_midi.PrettyMIDI(str(midi_path))
    acts = np.zeros((T, N_ORCH, N_PITCH), dtype=np.uint8)
    vels = np.zeros((T, N_ORCH), dtype=np.float32)
    for inst in pm.instruments:
        idx = INST_NAMES.index('timpani') if inst.is_drum else match(inst.name)
        if idx < 0: continue
        for note in inst.notes:
            f0 = max(0, int(note.start / HOP_S))
            f1 = max(f0+1, min(int(note.end / HOP_S), T))
            p = int(note.pitch); v = note.velocity / 127.0
            if 0 <= p < N_PITCH:
                acts[f0:f1, idx, p] = 1
                vels[f0:f1, idx] = np.maximum(vels[f0:f1, idx], v)
    return acts, vels

out_dir = Path(DATA_ROOT) / 'data/features/orch_pitch'
out_dir.mkdir(parents=True, exist_ok=True)
existing = len(list(out_dir.glob('*.npz')))
if existing < 170:
    print('Building orch_pitch targets from raw MIDI...')
    df = pd.read_csv(Path(DATA_ROOT) / 'data/features/meta/features_index.csv')
    manifest = pd.read_parquet(Path(DATA_ROOT) / 'data/processed/lop_manifest.parquet')
    midi_lut = dict(zip(manifest['pair_id'], manifest['orch_midi']))
    done = 0
    for _, row in df.iterrows():
        pair_id = row['pair_id']
        piano_npz = Path(DATA_ROOT) / row['piano_npz']
        slug = pair_id.replace('/', '__')
        out_path = out_dir / f'{slug}.npz'
        if not piano_npz.exists(): continue
        T = int(np.load(str(piano_npz), allow_pickle=True)['roll'].shape[0])
        raw = str(midi_lut.get(pair_id, ''))
        marker = 'LOP_database_06_09_17'
        if marker not in raw: continue
        midi_p = Path(DATA_ROOT) / 'data/raw' / raw[raw.index(marker):]
        if not midi_p.exists(): continue
        acts, vels = build_targets(midi_p, T)
        np.savez_compressed(str(out_path), activations=acts, velocities=vels)
        done += 1
    print(f'  ✅ Built {done} orch_pitch targets')
else:
    print(f'  ✅ orch_pitch targets on disk ({existing})')

# ── Build augmented targets if missing (needed for training only) ─────────
aug_out = Path(DATA_ROOT) / 'data/features/orch_pitch_aug'
aug_out.mkdir(parents=True, exist_ok=True)
aug_existing = len(list(aug_out.glob('*.npz')))
aug_index = Path(DATA_ROOT) / 'data/features/meta/features_index_aug.csv'

if aug_existing < 5000 and aug_index.exists():
    print('Building orch_pitch_aug targets (needed for training)...')
    aug_df = pd.read_csv(aug_index)
    done = 0
    for _, row in aug_df.iterrows():
        orig_id = row['orig_pair_id']
        pitch_shift = int(row['pitch_shift'])
        vel_scale = float(row['vel_scale'])
        aug_id = row['pair_id']
        slug = aug_id.replace('/', '__').replace('aug__', 'aug_')
        out_path = aug_out / f'{slug}.npz'
        if out_path.exists(): continue
        orig_slug = orig_id.replace('/', '__')
        orig_target = out_dir / f'{orig_slug}.npz'
        if not orig_target.exists(): continue
        d = np.load(str(orig_target))
        acts = d['activations'].astype(np.float32)
        vels_a = d['velocities'].astype(np.float32)
        if pitch_shift != 0:
            acts_s = np.zeros_like(acts)
            for p in range(N_PITCH):
                np_ = p + pitch_shift
                if 0 <= np_ < N_PITCH:
                    acts_s[:, :, np_] = acts[:, :, p]
            acts = acts_s
        vels_a = np.clip(vels_a * vel_scale, 0, 1)
        np.savez_compressed(str(out_path), activations=acts, velocities=vels_a)
        done += 1
    print(f'  ✅ Built {done} orch_pitch_aug targets')
elif aug_existing >= 5000:
    print(f'  ✅ orch_pitch_aug targets on disk ({aug_existing})')
else:
    print('  ℹ️  No aug index found — augmented targets will be built during training setup')

print('\n✅ Setup complete')

📂 Upload AI-for-Projective-Musical-Orchestration-main.zip


Saving AI-for-Projective-Musical-Orchestration-main.zip to AI-for-Projective-Musical-Orchestration-main.zip
  LOP zip: /content/AI-for-Projective-Musical-Orchestration-main.zip
📂 Upload orchestranet.zip


Saving orchestranet.zip to orchestranet.zip
  Code zip: /content/orchestranet.zip
  DATA_ROOT: /content/orchestration/AI-for-Projective-Musical-Orchestration-main
  CODE_ROOT: /content/orchestration/orchestranet

📂 Upload v6_best.pt  (skip with Cancel if training from scratch)


Saving v6_best.pt to v6_best.pt
  ✅ Checkpoint saved to /content/orchestration/orchestranet/runs/orchestranet_v6/best.pt
Building orch_pitch targets from raw MIDI...
  ✅ Built 178 orch_pitch targets
  ℹ️  No aug index found — augmented targets will be built during training setup

✅ Setup complete


Cell 2 — Inference

Orchestrate any MusicXML file using the trained model.

(Run Cell 1 first.)


In [ ]:
# ── Upload your MusicXML file ──────────────────────────────────────
import glob
from google.colab import files as colab_files

print('📂 Upload a MusicXML file (.xml or .musicxml)')
uploaded = colab_files.upload()
upload_name = list(uploaded.keys())[0]

# Find where Colab saved it (sometimes in CODE_ROOT)
candidates = glob.glob(f'/content/**/{upload_name}', recursive=True)
input_xml = candidates[0] if candidates else f'/content/{upload_name}'
print(f'Input: {input_xml}')

📂 Upload a MusicXML file (.xml or .musicxml)


Saving TM_full-test.musicxml to TM_full-test.musicxml
Input: /content/orchestration/orchestranet/TM_full-test.musicxml


In [ ]:
# ── Run orchestration ─────────────────────────────────────────────────────
import os, sys
sys.path.insert(0, CODE_ROOT)
sys.path.insert(0, os.path.join(CODE_ROOT, 'src'))

from symbolic_orchestrator import orchestrate_symbolic

base = os.path.splitext(os.path.basename(input_xml))[0]
output_xml = f'/content/{base}_orchestrated.xml'

orchestrate_symbolic(
    xml_path=input_xml,
    model_path=os.path.join(RUNS_DIR, 'best.pt'),
    output_path=output_xml,
    model_size='base',
    threshold=0.35,      # lower = more notes/instruments, higher = more selective
    device_str='cuda',
)

Loading model from /content/orchestration/orchestranet/runs/orchestranet_v6/best.pt...
Model loaded on cuda
Parsing /content/orchestration/orchestranet/TM_full-test.musicxml...
Found 3805 note/chord elements
Querying model for each note event...
  flute          : 203 elements
  oboe           : 675 elements
  clarinet       : 843 elements
  bassoon        : 411 elements
  horn           : 65 elements
  trumpet        : 13 elements
  trombone       : 595 elements
  tuba           : 137 elements
  violin_1       : 761 elements
  violin_2       : 787 elements
  viola          : 1596 elements
  cello          : 1525 elements
  double_bass    : 272 elements
  timpani        : 488 elements
  harp           : 20 elements

Building orchestral XML...
  flute          : 472 note-keys
  oboe           : 921 note-keys
  clarinet       : 1294 note-keys
  bassoon        : 636 note-keys
  horn           : 126 note-keys
  trumpet        : 45 note-keys
  trombone       : 900 note-keys
  tuba          

'/content/TM_full-test_orchestrated.xml'

In [ ]:
# ── Download the result ───────────────────────────────────────────────────
from google.colab import files as colab_files
colab_files.download(output_xml)
print(f'Downloaded: {output_xml}')
print('Open in MuseScore to view the orchestrated score.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: /content/TM_full-test_orchestrated.xml
Open in MuseScore to view the orchestrated score.


Cell 3 — Train from Scratch

Full 150-epoch training run (~14 hours on Colab T4 GPU).

Checkpoints autosave to your Google Drive every 5 minutes.

(Run Cell 1 first. You do not need to upload `v6_best.pt` for this.)


In [ ]:
# ── Connect Drive for checkpoint autosave ────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
DRIVE_SAVE_DIR = '/content/drive/MyDrive/OrchestraNet'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'Checkpoints will autosave to: {DRIVE_SAVE_DIR}')

In [ ]:
import shutil, threading, time, subprocess

stop_autosave = threading.Event()
def autosave_loop():
    last_saved = {'best.pt': 0, 'last.pt': 0}
    while not stop_autosave.is_set():
        time.sleep(5 * 60)
        for name, dname in [('best.pt', 'v6_best.pt'), ('last.pt', 'v6_last.pt')]:
            src = os.path.join(RUNS_DIR, name)
            if not os.path.exists(src): continue
            mtime = os.path.getmtime(src)
            if mtime > last_saved[name]:
                shutil.copy2(src, f'{DRIVE_SAVE_DIR}/{dname}')
                last_saved[name] = mtime
                print(f'💾 [{time.strftime("%H:%M")}] {name} → Drive')

t = threading.Thread(target=autosave_loop, daemon=True)
t.start()
print('✅ Autosave running')

proc = subprocess.Popen(
    ['python', 'scripts/train.py',
     '--data_root', DATA_ROOT,
     '--out_dir', RUNS_DIR,
     '--model_size', 'base',
     '--epochs', '150',
     '--batch_size', '16'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, cwd=CODE_ROOT
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

stop_autosave.set()
for name, dname in [('best.pt', 'v6_best.pt'), ('last.pt', 'v6_last.pt')]:
    src = os.path.join(RUNS_DIR, name)
    if os.path.exists(src):
        shutil.copy2(src, f'{DRIVE_SAVE_DIR}/{dname}')
        print(f'✅ {name} → Drive')

Cell 4 — Resume Training

Continue from `last.pt` after a Colab disconnect.

(Run Cell 1 first — upload `v6_best.pt` AND make sure you also have `v6_last.pt` saved from your previous session. Upload both when prompted in Cell 1 if needed, or restore them from Drive using the cell below.)


In [ ]:
# ── Optional: restore last.pt from Drive ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_SAVE_DIR = '/content/drive/MyDrive/OrchestraNet'

for name, dname in [('best.pt', 'v6_best.pt'), ('last.pt', 'v6_last.pt')]:
    src = f'{DRIVE_SAVE_DIR}/{dname}'
    dst = os.path.join(RUNS_DIR, name)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'✅ Restored {name} from Drive')
    else:
        print(f'⚠️  {dname} not found on Drive')

In [ ]:
import shutil, threading, time, subprocess

last_ckpt = os.path.join(RUNS_DIR, 'last.pt')
if not os.path.exists(last_ckpt):
    print('❌ last.pt not found — restore from Drive first (cell above)')
else:
    print(f'✅ Resuming from epoch in {last_ckpt}')

    stop_autosave = threading.Event()
    def autosave_loop():
        last_saved = {'best.pt': 0, 'last.pt': 0}
        while not stop_autosave.is_set():
            time.sleep(5 * 60)
            for name, dname in [('best.pt', 'v6_best.pt'), ('last.pt', 'v6_last.pt')]:
                src = os.path.join(RUNS_DIR, name)
                if not os.path.exists(src): continue
                mtime = os.path.getmtime(src)
                if mtime > last_saved[name]:
                    shutil.copy2(src, f'{DRIVE_SAVE_DIR}/{dname}')
                    last_saved[name] = mtime
                    print(f'💾 [{time.strftime("%H:%M")}] {name} → Drive')

    t = threading.Thread(target=autosave_loop, daemon=True)
    t.start()
    print('✅ Autosave running')

    proc = subprocess.Popen(
        ['python', 'scripts/train.py',
         '--data_root', DATA_ROOT,
         '--out_dir', RUNS_DIR,
         '--model_size', 'base',
         '--epochs', '150',
         '--batch_size', '16',
         '--resume', last_ckpt],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, cwd=CODE_ROOT
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

    stop_autosave.set()
    for name, dname in [('best.pt', 'v6_best.pt'), ('last.pt', 'v6_last.pt')]:
        src = os.path.join(RUNS_DIR, name)
        if os.path.exists(src):
            shutil.copy2(src, f'{DRIVE_SAVE_DIR}/{dname}')
            print(f'✅ {name} → Drive')

Cell 5 — Utilities


In [ ]:
# ── Check checkpoint info ─────────────────────────────────────────────────
import torch

for label, path in [('best.pt', os.path.join(RUNS_DIR,'best.pt')),
                    ('last.pt', os.path.join(RUNS_DIR,'last.pt'))]:
    if os.path.exists(path):
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        epoch = ckpt.get('epoch', '?')
        f1    = ckpt.get('best_f1', ckpt.get('val_f1', '?'))
        print(f'{label}: epoch={epoch}, val_f1={f1}')
    else:
        print(f'{label}: not found')

In [ ]:
# ── Try different thresholds and download all three ───────────────────────
import os, sys
sys.path.insert(0, CODE_ROOT)
sys.path.insert(0, os.path.join(CODE_ROOT, 'src'))
from symbolic_orchestrator import orchestrate_symbolic
from google.colab import files as colab_files

# input_xml must be set — run Cell 2 upload step first
for thresh in [0.25, 0.35, 0.45]:
    base = os.path.splitext(os.path.basename(input_xml))[0]
    out = f'/content/{base}_t{int(thresh*100)}.xml'
    print(f'\n--- threshold={thresh} ---')
    orchestrate_symbolic(
        xml_path=input_xml,
        model_path=os.path.join(RUNS_DIR, 'best.pt'),
        output_path=out,
        model_size='base',
        threshold=thresh,
        device_str='cuda',
    )
    colab_files.download(out)

In [ ]:
# ── Browse LOP piano XMLs in dataset ─────────────────────────────────────
import glob
xmls = glob.glob(f'{DATA_ROOT}/data/raw/**/*.xml', recursive=True)
piano_xmls = [x for x in xmls if 'piano' in x.lower()]
for x in piano_xmls[:20]:
    print(x)